Lab 4: LLMs and Prompt Engineering for Decision Support

Duration: 2 weeks [30 Jul - 13 Aug, 2026] Due Date: 13th August, 2026 Format: Jupyter Notebook / Google Colab + external APIs + GitHub version control Grading: This is a graded lab.

Student Name: Nii Sowah Student ID: 18682028

Objective

In the previous labs you trained models. In this lab you will use a model that someone else spent millions of dollars training — a Large Language Model (LLM) — and learn that getting good results out of one is an engineering discipline of its own: prompt engineering.

You will build a decision support system for a microfinance loan officer. Given a pile of free-text loan application letters, your system will:

    Summarize each application into a short, factual brief,
    Extract specific structured data points (JSON) that a downstream system could store,
    Produce a decision-support recommendation — while keeping the human firmly in the loop.

Just as importantly, you will evaluate the LLM's output for quality, reliability, and appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to make the final call?

Choosing an API provider

You need an LLM API with a free tier. Recommended options (pick ONE):
Provider 	Free tier 	Notes
Groq (recommended) 	Yes, generous 	OpenAI-compatible API, very fast, open models (Llama)
Google Gemini 	Yes 	google-generativeai package
Hugging Face Inference API 	Yes, limited 	Many open models
OpenAI / Anthropic 	Paid 	Fine if you already have credits

The notebook's example code uses the OpenAI-compatible chat format (works with Groq and OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is provider-agnostic.

Part 0: Repository and API-key setup

    Create a public repository named lab-4-llm-decision-support and save this notebook inside it.
    Sign up with your chosen provider and create an API key.
    NEVER hard-code or commit your API key. This is a graded requirement.
        Locally: put it in a .env file and add .env to .gitignore.
        Colab: use the Secrets panel (key icon) and read it with google.colab.userdata.
    Add a requirements.txt: openai python-dotenv pandas matplotlib.
    Commit and push after each Part — we will check for incremental commits.

    A leaked key in your commit history = resubmission + penalty. Keys can be scraped from public repos within minutes.


In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")
     

Client ready.


Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: messages and roles (system, user, assistant), and the generation parameters (temperature, max_tokens).
Part 1.1 — Your first API call

In [4]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
print(ask_llm("What is the capital of France?"))
# TODO: Print response.usage as well — how many tokens did your call consume?


The capital of France is Paris.


Student Reasoning — Anatomy of a call 1. What is the difference between the system and user roles? Give an example of something that belongs in each. 2. What is a token, roughly? Why do API providers bill per token rather than per request?

    Answer: A system role is used to set the behavior of the assistant, while a user role is used to provide input or ask questions. For example, a system message could be "You are a helpful assistant that provides concise answers," while a user message could be "What is the capital of France?" A token is roughly a word or a piece of a word, and API providers bill per token because it reflects the amount of computation and resources used to generate the response.


Part 1.2 — Temperature: the randomness dial

In [5]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
     
question = "Suggest a name for a savings product for market traders in Accra."

print("Temperature = 0.0 ")
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    print(f"\n Run {i+1} ")
    print(answer)

print("\n\n Temperature = 1.2")
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    print(f"\n Run {i+1} ")
    print(answer)

Temperature = 0.0 

 Run 1 
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who want to gather and save their earnings.
4. **Market Mobi**: This name incorporates "mobi", short for mobile, to suggest a convenient and accessible savings product.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, implying a sense of community and mutual support among market traders.
6. **Kae Dwa**: "Kae Dwa" means "good fortune" or "prosperity" in the Akan language, which could be an attractive name for a savings product.
7. **Accra Trader's Fund**: This name is straightforward and emphasizes the product'

Student Reasoning — Temperature What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

    Answer: [Double-click to edit]
